<a id="unit-u.91ea7ec9e54244b78987d2ad409223dd"></a>

<a id="partitions-and-buckets-for-the-incident-window"></a>

# Инцидент терезесінің бөлімдері мен bucket-тері

**5-модуль: Жүйені бақылау деректерін бөлімдеу және bucket-терге бөлу** · [Сабақ](course.md) · [Тест](quiz05_partition_and_bucket_choices.ipynb)

Төлем ақаулары басталғанда сұрақтар уақыт терезесі мен бірнеше сервиске тарылды: ақау басталғалы тапсырыстар, payment төлемдері. Тек сол терезе мен сервистердің дерегін оқитын сұрау ерте бітіп, машинаны басқалардың сұрауларына босатады. Doris әр кестені екі рет бөліп, осыны мүмкін етеді: уақытпен бөлімдерге, әр бөлімді tablet ретінде сақталатын bucket-терге.

Бұл зертханалық жұмыста экспорттаушы `otel_traces` кестесін қалай бөлгенін қарайсыз, `EXPLAIN` арқылы инцидент сұрауы қай бөлім мен tablet-ті оқитынын тексересіз және экспорттаушы RANDOM бөлуін span көшірмесіндегі сервис бойынша бөлумен салыстырасыз. Әр ұяшықты қайталауға болады.

Шамамен 35 минут жоспарлаңыз. Төлем ақауы қосылған 1-модульдің DOG стегі мен демосы қажет (1-зертханалық жұмыстың 5-бөлімі).

<a id="unit-u.90d0f2ff0268408f8caf3b5155a78eee"></a>

<a id="initialize-the-lab"></a>

### Зертханалық жұмысты инициализациялау

`lab.shell(...)` не `lab.sql(...)` қолданбас бұрын осы ұяшықты бір рет орындаңыз.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)


<a id="unit-u.9cf3041dc29d41f38d7a54570ca1124f"></a>

<a id="1-check-the-incidents-spans"></a>

## 1. Инцидент span-дарын тексеру

Бұл жұмыстың сұраулары инцидент терезесіне — алғашқы сәтсіз тапсырыстан қазіргі сәтке дейінгі уақытқа — қатысты. Алғашқы және соңғы сәтсіз тапсырысты табыңыз:

In [ ]:
incident = lab.wait_for_rows(
    """
    SELECT MIN(timestamp) AS first_failed_order,
           MAX(timestamp) AS latest_failed_order,
           COUNT(*) AS failed_orders
    FROM otel_traces
    WHERE service_name = 'checkout'
      AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
      AND status_code = 'STATUS_CODE_ERROR'
    HAVING COUNT(*) > 0
    """,
    title="The failed orders in otel_traces",
    hint="No order has failed. Check that the demo runs (lab.start_demo() in Module 1) and that paymentFailure "
         "is 50% (section 5 of Lab 1), wait a minute or two, then rerun this cell.",
    timeout_seconds=180,
)
INCIDENT_START = str(incident["first_failed_order"][0])
print("The incident window starts at", INCIDENT_START, "UTC")

<a id="unit-u.88f686e889854f86ae972f40947b0c9a"></a>

**Күтілетін нәтиже**

Бір жол: алғашқы және соңғы сәтсіз тапсырыстың UTC уақыты, алғашқыдан бері қанша тапсырыс сәтсіз болғаны — одан бері әр минутта шамамен біреуге артық. Ұяшық алғашқы уақытты `INCIDENT_START` ішінде сақтайды.

<a id="unit-u.af03c3e990454e98afaf32b3e6c5fe94"></a>

<a id="2-how-the-exporter-divided-otel_traces"></a>

## 2. Экспорттаушы `otel_traces` кестесін қалай бөлді

Экспорттаушы `otel_traces` кестесін `PARTITION BY RANGE(timestamp)`, күндік динамикалық бөлімдеу және `DISTRIBUTED BY RANDOM BUCKETS AUTO` арқылы құрған. **Бөлім** бөлім бағанының бір аралығының, мұнда span басталу уақытының бір UTC күнінің жолдарын сақтайды. Әр бөлім **bucket-терге** бөлінеді; Doris әр bucket-ті BE сақтайтын, көшіретін және сканерлейтін **tablet** ретінде сақтайды. Бөлімдерді тізіңіз:

In [ ]:
lab.sql("SHOW PARTITIONS FROM otel_traces", title="The partitions of otel_traces",
        columns=["PartitionName", "Range", "DistributionKey", "Buckets", "VisibleVersion"]);

<a id="unit-u.ca185a0ddc3d4a6ebc6e4c89ab675204"></a>

**Күтілетін нәтиже**

Күнге сай `p20261020` сияқты аталған бес бөлім: UTC бойынша бүгінге дейінгі үш күн, бүгін және ертең. `Range` әр бөлімнің алғашқы және кірмейтін соңғы уақытын береді. `DistributionKey` — `RANDOM`, әр бөлімнің `Buckets` мәні — 10. `VisibleVersion` жүктеулерді санайды: экспорттаушының әр Stream Load-ы нұсқа қосады, минутына бірнеше рет; сондықтан бүгінгі бөлімде ондағаннан мыңдағанға дейін нұсқа бар, ал нұсқасы 1 бөлім еш жол алмаған.

Бөлімдер әр 10 минутта жоспарлағыш қолданатын кестенің динамикалық бөлімдеуінен келеді:

In [ ]:
lab.sql("SHOW DYNAMIC PARTITION TABLES FROM otel", title="Dynamic partitioning of the exporter's tables",
        columns=["TableName", "TimeUnit", "Start", "End", "Buckets", "LastSchedulerTime"]);

<a id="unit-u.6002e63ad7f34a3c9e5873c421702c49"></a>

**Күтілетін нәтиже**

Экспорттаушы кестесіне бірден сегіз жол, барлығында `DAY`, `-7`, `1`, `10`. `End` = 1: түн ортасынан кейінгі алғашқы span-дарға орын болуы үшін жоспарлағыш ертеңгі бөлімді алдын ала құрады. `Start` = -7: бүгіннен 7 күннен астам бұрынғы күн бөлімдерін жояды, деректі қайта жазбай бір күндік телеметрияны бірден жояды. Экспорттаушы бұларды `history_days` және `create_history_days` баптауларынан орнатқан (2-модуль): `create_history_days: 3` — кесте өткен үш күнмен басталғанының себебі. `LastSchedulerTime` жоспарлағыштың соңғы 10 минут ішіндегі орындалуын көрсетеді.

Кестеде `BUCKETS AUTO` делінсе де `Buckets` — 10. Кестемен бірге құрылған бөлімдер `dynamic_partition.buckets` мәнін, әдепкі 10, алады. Әр кейінгі бөлімге Doris бұрынғы бөлімдердің өлшемінен, шамамен әр 5 GB сығылған дерекке бір bucket, баға береді; баға алдыңғы бөлім санының жартысынан аз болса, бұрынғы сан қалады. Курс бөлімдері мегабайттар, баға — 1, әр бөлім 10 bucket сақтайды.

<a id="unit-u.05a27f5eaea54d0babd136762b478135"></a>

<a id="3-which-partitions-a-query-reads"></a>

## 3. Сұрау қай бөлімдерді оқиды

Бөлім бағанын сүзетін сұрау тек сәйкес жолдар болуы мүмкін аралықтағы бөлімдерді оқиды: **бөлімдерді қысқарту**. `EXPLAIN` орындамай, жоспарды көрсетеді. `lab.explain_scan(...)` `EXPLAIN` орындап, әр кесте сканерлеуіне сүзгілерін, бар бөлімдер мен tablet-тердің қаншасын оқитынын көрсетеді; толық жоспар жиналған өрісте бар.

Инцидент терезесіндегі payment span-дарын санап, сол сұраудың жоспарын қараңыз:

In [ ]:
INCIDENT_QUERY = """
SELECT span_name,
       COUNT(*) AS spans,
       SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) AS failed
FROM {table}
WHERE service_name = 'payment'
  AND timestamp >= '{start}'
GROUP BY span_name
ORDER BY spans DESC
"""
lab.sql(INCIDENT_QUERY.format(table="otel_traces", start=INCIDENT_START),
        title="The payment service's spans since the first failed order")
random_scan = lab.explain_scan(INCIDENT_QUERY.format(table="otel_traces", start=INCIDENT_START),
                 title="Which partitions and tablets of otel_traces the query reads");

<a id="unit-u.a22e7d68b27e426b990a2f7dac8efe52"></a>

**Күтілетін нәтиже**

Сұрау: `grpc.oteldemo.PaymentService/Charge` — payment сервисінің әр төлем сұрауының сервер span-ы, шамамен жартысы сәтсіз; `charge` — оның ішіндегі сәтсіз болмайтын span. Кейін бір-екі span-ды жолдар болуы мүмкін, мысалы payment flagd-ден жалауша мәнін сұраған `grpc.flagd.evaluation.v1.Service/ResolveFloat`.

Жоспар: `otel.otel_traces` өзін оқитын (`reads` — `otel_traces`), екі сүзгісі де `predicates` астында тұрған бір сканерлеу. `partitions` — бүгінгі бөлім атауы бар `1/5`: `timestamp` сүзгісі басқаларын алып тастайды. Инцидент UTC түн ортасынан бұрын басталса, `2/5` — кеше мен бүгін. `tablets` — `10/10`: бір сервис сұралса да, оқылатын бөлімдердің барлық tablet-і оқылады. Себебін 4-бөлім түсіндіреді.

<a id="unit-u.990be8d1b247470cbb270027c3ae147a"></a>

Тағы екі жоспар қысқарту қайдан келетінін көрсетеді. Біріншісі кешегі күнді сұрайды, екіншісінде уақыт сүзгісі жоқ. Ұяшық кешегі UTC аралығын есептеп, сұрауға мәтінмен жазады, сондықтан Doris жоспарлауда мәндерді біледі:

In [ ]:
from datetime import datetime, timedelta, timezone

today = datetime.now(timezone.utc).date()
lab.explain_scan(f"""
SELECT COUNT(*) AS spans
FROM otel_traces
WHERE timestamp >= '{today - timedelta(days=1)}' AND timestamp < '{today}'
""", title="The plan of a query for yesterday")
lab.explain_scan("""
SELECT span_name, COUNT(*) AS spans
FROM otel_traces
WHERE service_name = 'payment'
GROUP BY span_name
""", title="The plan of a query without a filter on time");

<a id="unit-u.98f41d6c310044da8e47a53e7ac2e777"></a>

**Күтілетін нәтиже**

Кешегі сұрауда кесте сканерлеуі жоқ, карточка соны айтады: жолдары тек кешегі бөлімде болуы мүмкін, ал ол бос; сондықтан Doris оқымай жауап береді. (Демо кеше де жұмыс істесе, мысалы `lab.teardown(keep_data=True)` арқылы деректі сақтасаңыз, жоспар сол бөлімді оқиды.)

Уақыт сүзгісі жоқ сұрау да `1/5` бөлім оқиды: Doris бос бөлімдерді өткізеді, тек бүгінгісінде жолдар бар. Бір апталық телеметрияда бұл сұрау барлық сегіз бөлімді, инцидент терезесінің сұрауы әлі бір не екі бөлімді оқыр еді. Өндірісте уақытпен қысқарту — сұрауды кішірейтетін алғашқы қадам: әрдайым бөлім бағанын Doris жоспарлауда білетін мәндермен, мысалы есептеген уақытыңыз не `UTC_TIMESTAMP() - INTERVAL 1 HOUR` арқылы сүзіңіз.

<a id="unit-u.4b570e3791c64dd3a0fe55593432c5b1"></a>

<a id="4-buckets-random-or-hash"></a>

## 4. Bucket-тер: RANDOM не HASH

Бөлім ішінде тарату әр жолдың қай bucket-ке баратынын анықтайды:

- `DISTRIBUTED BY HASH(column)`: баған мәнінің хешімен. Мәні бірдей барлық жол бір bucket-ке түседі, сондықтан `column = value` сүзгісі біреуін оқиды: **bucket-терді қысқарту**. Bucket-тер мәндердей ғана біркелкі: жолы көп мән bucket-ін үлкейтеді.
- `DISTRIBUTED BY RANDOM`: бағансыз. Жүктеу әр жол пакетін бір bucket-ке, келесісін келесісіне жібереді; сондықтан шамамен біркелкі, бірақ еш сүзгі bucket-терді қысқарта алмайды.

Экспорттаушы RANDOM таңдады. `otel_traces` әр tablet жолдарын санаңыз. `lab.tablet_rows(...)` дерегі бар әр бөлімнің tablet-терін `SHOW TABLETS` арқылы тізіп, әрқайсысының жолын `SELECT COUNT(*) FROM ... TABLET(<id>)` арқылы санайды:

In [ ]:
random_distribution = lab.tablet_rows("otel_traces");

<a id="unit-u.01bb5d088b9d405bb4d5a3219be42b6a"></a>

**Күтілетін нәтиже**

Алдымен командалар, содан кейін бүгінгі бөлімнің он tablet-і; әрқайсысында жолдарының шамамен оннан бірі: әр tablet үшін `percent_of_partition` шамамен 7–13 арасында. Пакет көбейген сайын үлестер 10-ға жақындайды.

<a id="unit-u.243c8cac5b434861b02eab1496575fca"></a>

Енді сол span-дарды сервиспен бөліңіз. Келесі ұяшық `otel_traces` негізгі бағандарының көшірмесі `spans_by_service` кестесін `DISTRIBUTED BY HASH(service_name) BUCKETS 10` арқылы құрып, оған `otel_traces` кестесінің барлық span-ын жүктейді. Кесте де күнмен, 5-бөлім түсіндіретін `AUTO PARTITION` арқылы бөлімделеді.

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS spans_by_service (
    timestamp      DATETIME(6) NOT NULL,
    service_name   VARCHAR(200),
    trace_id       VARCHAR(200),
    span_id        STRING,
    parent_span_id STRING,
    span_name      STRING,
    span_kind      STRING,
    end_time       DATETIME(6),
    duration       BIGINT,
    status_code    STRING,
    status_message STRING
)
DUPLICATE KEY(timestamp, service_name)
AUTO PARTITION BY RANGE (date_trunc(timestamp, 'day')) ()
DISTRIBUTED BY HASH(service_name) BUCKETS 10
PROPERTIES ("partition.retention_count" = "7")
""", title="Create spans_by_service, bucketed by service")
lab.execute("TRUNCATE TABLE spans_by_service", title="Empty the table, so that the cell can be rerun")
lab.execute("""
INSERT INTO spans_by_service
SELECT timestamp, service_name, trace_id, span_id, parent_span_id, span_name, span_kind, end_time, duration, status_code, status_message
FROM otel_traces
""", title="Copy the spans of otel_traces")
service_distribution = lab.tablet_rows("spans_by_service");

<a id="unit-u.81aee8b2098e47adb53373909e737e38"></a>

**Күтілетін нәтиже**

Кірістіру әр span-ға бір әсер еткен жол, ондаған мың не одан көп жол хабарлайды. Содан кейін біркелкілігі нашар он tablet. `frontend-proxy` мен `frontend` сияқты ең көп жұмыс істейтін сервистер tablet-терінде жолдардың төрттен бірі не одан көбі бар; бір-екі tablet-те аз не жоқ, payment tablet-і кішкентай. Қай сервистер бірге түсетіні тек хештеріне байланысты. Бөлімнің көп бөлігін сақтайтын tablet оның жүктеуін, compaction-ын және әр сканерлеуін басқалардан баяулатады: бөлім ең үлкен tablet-індей ғана жылдам.

<a id="unit-u.2cf6670e7fd94bef91ddb123c2ae7837"></a>

Көшірмедегі инцидент сұрауының жоспарын қараңыз:

In [ ]:
service_scan = lab.explain_scan(INCIDENT_QUERY.format(table="spans_by_service", start=INCIDENT_START),
                 title="Which partitions and tablets of spans_by_service the query reads");

<a id="unit-u.9f8cdc90d7e54a83b725bb581a043749"></a>

**Күтілетін нәтиже**

`otel.spans_by_service` бір сканерлеуі, `partitions` — `1/1`, `tablets` — `1/10`: `service_name` HASH бөлуінде Doris `'payment'` bucket-ін есептеп, тек сол tablet-ті оқиды.

Демек bucket-ке бөлу — қысқарту мен теңгерім арасындағы таңдау. Көп сұрау бағанды `=` не `IN` арқылы сүзсе және оның мәндері жолдарды біркелкі таратса, сол бағанмен бөліңіз; бөлек мәні көп, басым мәні жоқ баған ең жақсы таратады. Бастапқы телеметрия көбіне уақытпен оқылып, әртүрлі ізделеді, сервистері өте біркелкі емес. Сондықтан экспорттаушы RANDOM қолданады: біркелкі tablet-тер, ал бір сервис жолдарын `service_name` кері индексі сияқты басқа тәсілдер табады (6-модуль).

<a id="unit-u.ff755829f2cc4a89afec64fd473bb303"></a>

<a id="5-auto-partition-partitions-that-the-data-creates"></a>

## 5. AUTO PARTITION: дерек құратын бөлімдер

`spans_by_service` кестесі `AUTO PARTITION BY RANGE (date_trunc(timestamp, 'day'))` арқылы бөлімделеді: ол бөлімсіз құрылған, Doris жүктелген жол тиесілі әр күнге бөлім құрды. Оларды тізіңіз:

In [ ]:
lab.sql("SHOW PARTITIONS FROM spans_by_service", title="The partitions of spans_by_service",
        columns=["PartitionName", "Range", "Buckets", "VisibleVersion"]);

<a id="unit-u.c96d0d5cf3c6479e90f834ec931338f8"></a>

**Күтілетін нәтиже**

Көшірілген span-дардың әр күніне бір бөлім, әдетте біреу — бүгінгі, атауы `p20261020000000` сияқты, 10 bucket-пен. `otel_traces` динамикалық бөлімдерінен өзгеше, бос бөлім жоқ: дерексіз өткен күндер де, ертеңгі жол келмейінше ертеңгі бөлім де жоқ.

AUTO PARTITION ескі бөлімдерді өзі жоймайды. Кестенің `"partition.retention_count" = "7"` қасиеті Doris-тен бүгінге дейінгі ең жаңа жеті бөлімді сақтап, ескілерін жоюды сұрайды. Оны динамикалық бөлімдеудегідей жоспарлағыш әр 10 минутта қолданады, сондықтан кестеде өткен жеті күннен көп дерек болғанда ғана әсер етеді. `SHOW CREATE TABLE spans_by_service` қасиетті көрсетеді. AUTO PARTITION кестесінде динамикалық бөлімдеу де болуы мүмкін, бірақ `partition.retention_count` қасиетімен бірге емес.

<a id="unit-u.b0066ec3338240aea77f7e1e9a22f3c4"></a>

<a id="6-exercise-bucket-the-spans-by-trace"></a>

## 6. Жаттығу: span-дарды трасса бойынша бөлу

Инженер 3-модульдегідей трассаны ашқанда, Doris бір `trace_id` барлық span-дарын іздейді. Бағандары, бөлімдеуі және `partition.retention_count = 7` мәні `spans_by_service` сияқты, бірақ `DISTRIBUTED BY HASH(trace_id) BUCKETS 10` болатын `spans_by_trace` құрып, оған `otel_traces` span-дарын жүктеңіз. Жолдар tablet-терге қаншалықты біркелкі таралған? Соңғы сәтсіз тапсырыстың трасса ID-ін алып, оның span-дары сұрауы қанша tablet оқитынын `lab.explain_scan(...)` арқылы тексеріңіз.

In [ ]:
# Your code here

<a id="unit-u.755bf2573fc449d98a529f7e6fd72222"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған. Салыстыру үшін ашыңыз немесе орындаңыз.

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS spans_by_trace (
    timestamp      DATETIME(6) NOT NULL,
    service_name   VARCHAR(200),
    trace_id       VARCHAR(200),
    span_id        STRING,
    parent_span_id STRING,
    span_name      STRING,
    span_kind      STRING,
    end_time       DATETIME(6),
    duration       BIGINT,
    status_code    STRING,
    status_message STRING
)
DUPLICATE KEY(timestamp, service_name)
AUTO PARTITION BY RANGE (date_trunc(timestamp, 'day')) ()
DISTRIBUTED BY HASH(trace_id) BUCKETS 10
PROPERTIES ("partition.retention_count" = "7")
""", title="Create spans_by_trace, bucketed by trace")
lab.execute("TRUNCATE TABLE spans_by_trace", title="Empty the table, so that the cell can be rerun")
lab.execute("""
INSERT INTO spans_by_trace
SELECT timestamp, service_name, trace_id, span_id, parent_span_id, span_name, span_kind, end_time, duration, status_code, status_message
FROM otel_traces
""", title="Copy the spans of otel_traces")
lab.tablet_rows("spans_by_trace")

failed_trace = lab.sql("""
SELECT trace_id
FROM spans_by_trace
WHERE service_name = 'checkout'
  AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
  AND status_code = 'STATUS_CODE_ERROR'
ORDER BY timestamp DESC
LIMIT 1
""", title="The trace of the latest failed order")["trace_id"][0]
lab.explain_scan(f"SELECT * FROM spans_by_trace WHERE trace_id = '{failed_trace}'",
                 title="Which tablets a lookup of one trace reads");

<a id="unit-u.df71da7ec3a5435db953818970cb5300"></a>

**Күтілетін нәтиже**

Әрқайсысында жолдардың шамамен оннан бірі бар он tablet: трасса ID-лері кездейсоқ, бір трассада бірнеше ондағаннан артық span жоқ. Бір трасса іздеуі `1/10` tablet, сол трассаның барлық span-ы бар біреуін оқиды. Сондықтан көбіне трассалап оқылатын, мысалы қолданба трассаларының кестесі трасса ID-імен бөлініп, біркелкі tablet пен қысқартуды қатар ала алады; 7-модульдегі Litefuse солай істейді. Экспорттаушының `otel_traces` кестесі әртүрлі оқылады, сондықтан RANDOM болып қалады.

<a id="unit-u.bd6218e831d24425a69f854a02d0ad47"></a>

<a id="7-compare-the-three-bucket-choices"></a>

## 7. Үш bucket таңдауын салыстыру

Жаттығуда `spans_by_trace` құрғаннан кейін нәтижелерді біріктіру үшін осы ұяшықты орындаңыз. Ол 3–4-бөлімнің RANDOM және service-HASH бақылауларын пайдаланып, трасса көшірмесін өлшейді. Жаттығу жауабындағы айнымалы атауларына тәуелді емес. Кірістер әр сәтте өлшенген, сондықтан демо жалғасқанда жалпы жол сандары ерекшеленуі мүмкін.

In [ ]:
import pandas as pd
from IPython.display import display

trace_lookup_id = lab.query("""
SELECT trace_id FROM spans_by_trace
WHERE service_name = 'checkout'
  AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
  AND status_code = 'STATUS_CODE_ERROR'
ORDER BY timestamp DESC LIMIT 1
""")["trace_id"].iloc[0]
trace_distribution = lab.tablet_rows("spans_by_trace")
trace_scan = lab.explain_scan(f"SELECT * FROM spans_by_trace WHERE trace_id = '{trace_lookup_id}'",
                              title="The trace lookup used in the comparison")

bucket_comparison = pd.DataFrame([
    {"bucketing": name, "sampled_rows": int(distribution["rows"].sum()),
     "smallest_tablet_share_percent": distribution["percent_of_partition"].min(),
     "largest_tablet_share_percent": distribution["percent_of_partition"].max(),
     "lookup": lookup, "tablets_read": scan["tablets"].iloc[0]}
    for name, distribution, scan, lookup in (
        ("RANDOM", random_distribution, random_scan, "payment since the incident began"),
        ("HASH(service_name)", service_distribution, service_scan, "payment since the incident began"),
        ("HASH(trace_id)", trace_distribution, trace_scan, "one failed order's trace"),
    )
])
display(bucket_comparison)

<a id="unit-u.af610e42564848e187f26e49a44ec481"></a>

**Күтілетін нәтиже**

Осы орындалудан үш қорытынды жол: RANDOM және trace-HASH tablet үлестері 10%-ға жақын, service-HASH ең үлкен tablet-і әлдеқайда ірі. Payment іздеуі таңдалған барлық RANDOM tablet-терін және әр таңдалған бөлімнің бір service-HASH tablet-ін оқиды; трасса іздеуі әр таңдалған бөлімнің бір trace-HASH tablet-ін оқиды. Үлестер — әр дерегі бар бөлім ішіндегі пайыздар; оқылатын tablet-тердің арақатынасы бекітілген мысал санынан емес, нақты жоспарлардан алынған. Bucket кілтін қажетті іздеулерге де, тудыратын теңгеріміне де қарай таңдаңыз.

<a id="unit-u.ac924ba4d7994186a15891152b64a05a"></a>

<a id="you-are-done-when"></a>

## Аяқтау шарттары

- `otel_traces` ішінде неше бөлім барын, неге және инцидент сұрауы қайсысын оқитынын айта аласыз;
- RANDOM кестесі бөлімнің әр tablet-ін, ал сервиспен бөлінгені тек сол сервистің tablet-ін оқитынын, бірақ tablet-тері біркелкі болмайтынын көрдіңіз;
- AUTO PARTITION қандай бөлімдер құратынын, `partition.retention_count` пен динамикалық бөлімдеудің `start` мәні нені жоятынын айта аласыз.

Көшірмелер кішкентай, кейінгі модульдер оларды қолданбайды. Жою үшін `lab.execute(...)` не MySQL клиентімен `DROP TABLE IF EXISTS spans_by_service` және `DROP TABLE IF EXISTS spans_by_trace` орындаңыз.

<a id="unit-u.b135d1a5143a49ac90bb93e9c677f95f"></a>

<a id="next"></a>

## Әрі қарай

[5-тест: Жүйені бақылау деректерін бөлімдеу және bucket-терге бөлу](quiz05_partition_and_bucket_choices.ipynb) файлын ашып, [6-модуль: Кесте қасиеттері, материалданған көріністер және индекстер](../module06-property-mv-index/course.md) бөліміне өтіңіз.